# Finnish ASR – data preparation

Team Tokens · COE025 Project 3

Builds one cleaned Finnish speech dataset from three sources:

| Source | Splits used |
|---|---|
| Mozilla Common Voice Scripted Speech 27.0 – Finnish (CC0) | train, dev, test |
| Aalto Finnish Parliament ASR Corpus, dev/eval (Zenodo 4581941, CC BY 4.0) | dev, test |
| Google FLEURS fi_fi (CC BY 4.0) | test |

Output (one `DatasetDict`, pushed to a public Hugging Face dataset: https://huggingface.co/datasets/zehraoz36/team-tokens-finnish-asr-data):

| Split | Content |
|---|---|
| `train` | Common Voice train + validated clips (dev/test speakers and sentences excluded) |
| `dev_cv`, `dev_parl` | Common Voice dev, Parliament dev |
| `test_cv`, `test_parl`, `test_fleurs` | three test sets, always reported separately |

Every clip: 16 kHz mono WAV, 0.5–30 s, non-empty normalized text.
The CSV lists (no audio) are saved to `data/` for the GitHub repo.

**Before running:** Runtime → Change runtime type → CPU is enough (no GPU needed).
Upload the Common Voice `.tar.gz` to the Files panel (left) or to Google Drive (`MyDrive/`).

In [14]:
!pip install -q "datasets>=3" soundfile pandas huggingface_hub

## 1. Settings

In [16]:
HF_DATASET = "zehraoz36/team-tokens-finnish-asr-data"   # change to your Hugging Face username/name
WORK = "/content/work"          # raw downloads and extracted files
AUDIO = "/content/audio16k"     # converted 16 kHz WAV files
OUT_CSV = "/content/data"       # CSV lists for the GitHub repo

MIN_SEC, MAX_SEC = 0.5, 30.0    # Whisper processes at most 30 s per input
PARL_URL = "https://zenodo.org/records/4581941/files/fi-parl-asr-dev-eval.zip?download=1"
FLEURS_PARQUET = "hf://datasets/google/fleurs@refs/convert/parquet/fi_fi/test/*.parquet"

import os, glob, re, csv, io, subprocess, tarfile, zipfile, unicodedata
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import soundfile as sf

for d in (WORK, AUDIO, OUT_CSV):
    os.makedirs(d, exist_ok=True)

## 2. Text normalization

One policy for every source and every model (lecture slide 45).
Lowercase, punctuation → space, keep Finnish letters (å ä ö) and digits, collapse spaces.

In [18]:
_BRACKETS = re.compile(r"\[[^\]]*\]|<[^>]*>|\(\S*\)\s*$")   # [noise], <unk>, trailing (utt-id)
# Use this SAME function on model predictions before WER/CER.

def normalize(text: str) -> str:
    text = unicodedata.normalize("NFKC", str(text or ""))
    text = _BRACKETS.sub(" ", text)
    text = text.lower()
    text = re.sub(r"(?<=\d)[ .,](?=\d{3}\b)", "", text)   # 1 500 / 1.500 -> 1500
    text = re.sub(r"[^\w\s]", " ", text)                     # punctuation (also - and ') -> space
    text = text.replace("_", " ")
    return re.sub(r"\s+", " ", text).strip()

for s in ["Hyvää päivää!", "EU-maiden (noin 1 500) edustajat.", "Arvoisa puhemies, <unk> kiitos."]:
    print(repr(s), "->", repr(normalize(s)))

'Hyvää päivää!' -> 'hyvää päivää'
'EU-maiden (noin 1 500) edustajat.' -> 'eu maiden noin 1500 edustajat'
'Arvoisa puhemies, <unk> kiitos.' -> 'arvoisa puhemies kiitos'


## 3. Common Voice (train / dev / test)

In [20]:
# find the uploaded archive (Files panel or Google Drive)
cands = glob.glob("/content/*.tar.gz")
if not cands:
    from google.colab import drive
    drive.mount("/content/drive")
    cands = glob.glob("/content/drive/MyDrive/**/*cv*fi*.tar.gz", recursive=True) + \
            glob.glob("/content/drive/MyDrive/**/common-voice*.tar.gz", recursive=True)
assert cands, "Common Voice .tar.gz not found: upload it to /content or to Google Drive"
CV_TAR = cands[0]
print("Using", CV_TAR)

!mkdir -p {WORK}/cv && tar -xzf "{CV_TAR}" -C {WORK}/cv

cv_dir = os.path.dirname(glob.glob(f"{WORK}/cv/**/train.tsv", recursive=True)[0])
print("Common Voice folder:", cv_dir)
print(sorted(os.listdir(cv_dir)))

Using /content/1789492402085-cv-corpus-27.0-2026-09-11-fi.tar.gz
Common Voice folder: /content/work/cv/cv-corpus-27.0-2026-09-11/fi
['README.md', 'clip_durations.tsv', 'clips', 'dev.tsv', 'invalidated.tsv', 'other.tsv', 'reported.tsv', 'test.tsv', 'train.tsv', 'unvalidated_sentences.tsv', 'validated.tsv', 'validated_sentences.tsv']


In [21]:
def read_cv(split):
    df = pd.read_csv(f"{cv_dir}/{split}.tsv", sep="\t", quoting=csv.QUOTE_NONE, dtype=str)
    return pd.DataFrame({
        "id": "cv_" + df["path"].str.replace(".mp3", "", regex=False),
        "src_path": cv_dir + "/clips/" + df["path"],
        "raw_text": df["sentence"],
        "speaker": df["client_id"],
        "source": "common_voice",
    })

cv = {s: read_cv(s) for s in ["train", "dev", "test"]}
for s, df in cv.items():
    print(f"{s:5s} {len(df):6d} clips  {df.speaker.nunique():4d} speakers")

# Common Voice official splits should not share speakers
print("speaker overlap train/test:", len(set(cv["train"].speaker) & set(cv["test"].speaker)))

train   2067 clips    10 speakers
dev     1827 clips    27 speakers
test    1833 clips   187 speakers
speaker overlap train/test: 0


## 4. Aalto Finnish Parliament (dev / eval)

In [22]:
zip_path = f"{WORK}/fi-parl-asr-dev-eval.zip"
if not os.path.exists(zip_path):
    !wget -q --show-progress -O {zip_path} "{PARL_URL}"
with zipfile.ZipFile(zip_path) as z:
    z.extractall(f"{WORK}/parl")

def read_trn(path):
    with open(path, encoding="utf-8", errors="replace") as f:
        return " ".join(line.strip() for line in f)

rows = []
for wav in glob.glob(f"{WORK}/parl/**/*.wav", recursive=True):
    trn = os.path.splitext(wav)[0] + ".trn"
    if not os.path.exists(trn):
        continue
    rows.append({"id": "parl_" + os.path.splitext(os.path.basename(wav))[0], "src_path": wav,
                 "raw_text": read_trn(trn), "speaker": os.path.basename(os.path.dirname(wav)),
                 "source": "parliament"})
parl = pd.DataFrame(rows)
print(parl.raw_text.head(5).tolist())   # check the transcript format by eye

In [24]:
# dev/eval membership comes from the official .list files (seen = speaker also in the big train set)
base = f"{WORK}/parl/fi-parliament-asr"
lookup = {}
for sub in ["seen", "unseen"]:
    for part in ["dev", "eval"]:
        with open(f"{base}/{sub}_{part}.list", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    key = os.path.splitext(os.path.basename(line.split()[0]))[0]
                    lookup[key] = (part, sub)

parl["key"] = parl.src_path.map(lambda p: os.path.splitext(os.path.basename(p))[0])
parl["split"] = parl.key.map(lambda k: lookup.get(k, ("?", ""))[0])
parl["subset"] = parl.key.map(lambda k: lookup.get(k, ("?", ""))[1])
print(pd.crosstab(parl.subset, parl.split))
parl["speaker"] = parl.key.str.rsplit("_", n=1).str[0]   # file name = <speaker>_<number>
print(parl.groupby("split").speaker.nunique())

split   dev  eval
subset           
seen    843   966
unseen  954   962
seen/anne_mari_virolainen_04297.wav
seen/anne_mari_virolainen_04299.wav
seen/anne_mari_virolainen_04300.wav


## 5. FLEURS (test)

In [26]:
from datasets import load_dataset, Audio
try:
    fl = load_dataset("parquet", data_files={"test": FLEURS_PARQUET}, split="test")
except Exception as e:
    print("parquet export failed:", e)
    fl = load_dataset("google/fleurs", "fi_fi", split="test", trust_remote_code=True)
fl = fl.cast_column("audio", Audio(decode=False))

os.makedirs(f"{WORK}/fleurs", exist_ok=True)
rows = []
for i, ex in enumerate(fl):
    p = f"{WORK}/fleurs/{i:05d}.wav"
    with open(p, "wb") as f:
        f.write(ex["audio"]["bytes"])
    rows.append({"id": f"fleurs_{i:05d}", "src_path": p,
                 "raw_text": ex.get("raw_transcription") or ex["transcription"],
                 "speaker": str(ex.get("id")), "source": "fleurs"})
fleurs = pd.DataFrame(rows)
print(len(fleurs), "FLEURS test clips")

fi_fi/test/0000.parquet: reconstructing file:   0%|          |  0.00B /  745MB            

fi_fi/test/0000.parquet: downloading bytes:           |  0.00B            

Generating test split: 0 examples [00:00, ? examples/s]

918 FLEURS test clips


## 6. Combine, convert to 16 kHz mono WAV, filter

Split names follow the plan: one training set, two dev sets, three separate test sets.

In [27]:
frames = [
    cv["train"].assign(split="train"),
    cv["dev"].assign(split="dev_cv"),
    cv["test"].assign(split="test_cv"),
    parl[parl.split == "dev"].assign(split="dev_parl"),
    parl[parl.split == "eval"].assign(split="test_parl"),
    fleurs.assign(split="test_fleurs"),
]
allc = pd.concat(frames, ignore_index=True)
allc["text"] = allc["raw_text"].map(normalize)
print(allc.groupby("split").size())

def convert(row):
    out = f"{AUDIO}/{row.id}.wav"
    if not os.path.exists(out):
        r = subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-i", row.src_path,
                            "-ac", "1", "-ar", "16000", "-sample_fmt", "s16", out])
        if r.returncode != 0:
            return out, None
    try:
        return out, sf.info(out).duration
    except Exception:
        return out, None

with ThreadPoolExecutor(max_workers=os.cpu_count() * 2) as ex:
    res = list(ex.map(convert, allc.itertuples(index=False)))
allc["audio"] = [r[0] for r in res]
allc["duration"] = [r[1] for r in res]

split
dev_cv         1827
dev_parl       1797
test_cv        1833
test_fleurs     918
test_parl      1928
train          2067
dtype: int64


In [28]:
before = allc.groupby("split").size()
bad_audio = allc.duration.isna()
empty = allc.text.str.len() == 0
bad_len = ~allc.duration.between(MIN_SEC, MAX_SEC)
dupe = allc.duplicated(["split", "id"])
clean = allc[~(bad_audio | empty | bad_len | dupe)].reset_index(drop=True)

report = pd.DataFrame({
    "before": before,
    "after": clean.groupby("split").size(),
    "hours": (clean.groupby("split").duration.sum() / 3600).round(2),
    "speakers": clean.groupby("split").speaker.nunique(),
}).loc[["train", "dev_cv", "dev_parl", "test_cv", "test_parl", "test_fleurs"]]
print("removed: bad audio", int(bad_audio.sum()), "| empty text", int(empty.sum()),
      "| outside 0.5-30 s", int((bad_len & ~bad_audio).sum()), "| duplicates", int(dupe.sum()))
report

removed: bad audio 0 | empty text 0 | outside 0.5-30 s 40 | duplicates 0


,before,after,hours,speakers
split,,,,
train,2067,2067,2.55,10
dev_cv,1827,1827,2.32,27
dev_parl,1797,1774,5.16,19
test_cv,1833,1833,2.61,187
test_parl,1928,1913,5.55,21
test_fleurs,918,916,3.29,348


In [29]:
# Common Voice fi train is small: add validated clips whose speakers and sentences are not in dev/test
val = pd.read_csv(f"{cv_dir}/validated.tsv", sep="\t", quoting=csv.QUOTE_NONE, dtype=str)
held_spk = set(cv["dev"].speaker) | set(cv["test"].speaker)
held_txt = set(clean[clean.split != "train"].text)
extra = pd.DataFrame({
    "id": "cv_" + val["path"].str.replace(".mp3", "", regex=False),
    "src_path": cv_dir + "/clips/" + val["path"],
    "raw_text": val["sentence"], "speaker": val["client_id"],
    "source": "common_voice", "split": "train"})
extra["text"] = extra.raw_text.map(normalize)
extra = extra[~extra.speaker.isin(held_spk) & ~extra.id.isin(clean.id) & ~extra.text.isin(held_txt)]
print("added clips:", len(extra), "| speakers:", extra.speaker.nunique())

with ThreadPoolExecutor(max_workers=os.cpu_count() * 2) as ex:
    res = list(ex.map(convert, extra.itertuples(index=False)))
extra["audio"], extra["duration"] = [r[0] for r in res], [r[1] for r in res]
extra = extra[extra.duration.between(MIN_SEC, MAX_SEC) & (extra.text.str.len() > 0)]

clean = pd.concat([clean, extra], ignore_index=True)
report = pd.DataFrame({
    "clips": clean.groupby("split").size(),
    "hours": (clean.groupby("split").duration.sum() / 3600).round(2),
    "speakers": clean.groupby("split").speaker.nunique()}).loc[report.index]
report.loc["test_fleurs", "speakers"] = None
report

eklenecek klip: 1703 | yeni konuşmacı: 8


,clips,hours,speakers
split,,,
train,3770,4.65,10.0
dev_cv,1827,2.32,27.0
dev_parl,1774,5.16,19.0
test_cv,1833,2.61,187.0
test_parl,1913,5.55,21.0
test_fleurs,916,3.29,NaN


## 7. Save CSV lists (for GitHub `data/`)

In [30]:
cols = ["id", "split", "source", "speaker", "duration", "text", "raw_text"]
for split, df in clean.groupby("split"):
    df[cols].assign(duration=df.duration.round(3)).to_csv(f"{OUT_CSV}/{split}.csv", index=False)
report.to_csv(f"{OUT_CSV}/summary.csv")
print(sorted(os.listdir(OUT_CSV)))

['dev_cv.csv', 'dev_parl.csv', 'summary.csv', 'test_cv.csv', 'test_fleurs.csv', 'test_parl.csv', 'train.csv']


## 8. Push the audio dataset to Hugging Face 

Log in with a Hugging Face account that has write access (browser login or a write token).

In [31]:
from huggingface_hub import login
login()   # paste the write token

In [32]:
from datasets import Dataset, DatasetDict, Audio
dd = DatasetDict({
    split: Dataset.from_pandas(df[["audio"] + cols].reset_index(drop=True), preserve_index=False)
                 .cast_column("audio", Audio(sampling_rate=16000))
    for split, df in clean.groupby("split")
})
print(dd)
dd.push_to_hub(HF_DATASET, private=False)
print("https://huggingface.co/datasets/" + HF_DATASET)

DatasetDict({
    dev_cv: Dataset({
        features: ['audio', 'id', 'split', 'source', 'speaker', 'duration', 'text', 'raw_text'],
        num_rows: 1827
    })
    dev_parl: Dataset({
        features: ['audio', 'id', 'split', 'source', 'speaker', 'duration', 'text', 'raw_text'],
        num_rows: 1774
    })
    test_cv: Dataset({
        features: ['audio', 'id', 'split', 'source', 'speaker', 'duration', 'text', 'raw_text'],
        num_rows: 1833
    })
    test_fleurs: Dataset({
        features: ['audio', 'id', 'split', 'source', 'speaker', 'duration', 'text', 'raw_text'],
        num_rows: 916
    })
    test_parl: Dataset({
        features: ['audio', 'id', 'split', 'source', 'speaker', 'duration', 'text', 'raw_text'],
        num_rows: 1913
    })
    train: Dataset({
        features: ['audio', 'id', 'split', 'source', 'speaker', 'duration', 'text', 'raw_text'],
        num_rows: 3770
    })
})


Setting num_proc from 1 back to 1 for the dev_cv split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Map:   0%|          | 0/1827 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

... (55 outputs left out)

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmpabvjc86b.parquet    :   0%|          |  669kB /  270MB            

Validating                    :           |   0%            

https://huggingface.co/datasets/zehraoz36/team-tokens-finnish-asr-data


## 9. Download the CSV lists

Upload the files to the GitHub repo under `data/`. Teammates load the audio with:

```python
from datasets import load_dataset
ds = load_dataset("zehraoz36/team-tokens-finnish-asr-data")
```

In [33]:
!cd /content && zip -qr data_csv.zip data
from google.colab import files
files.download("/content/data_csv.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>